# Dedicated Phase D Continuation for Qwen2.5-VL

This notebook keeps the previous larger-split results untouched, attaches the best published Phase C larger-split checkpoint, and runs only Phase D into a separate output root.


In [ ]:
import glob, json, os, shutil, subprocess, tarfile

WORKDIR = "/kaggle/working/RL_GSPO_Qwen2.5VLM"
VENV_DIR = "/tmp/rl-gspo-venv"
VENV_PYTHON = f"{VENV_DIR}/bin/python"


def find_code_dataset_root():
    matches = []
    for root, _, files in os.walk("/kaggle/input"):
        if "rl_gspo_qwen2_5vlm_test3.py" in files:
            matches.append(root)
    if not matches:
        raise FileNotFoundError(f"Could not find attached code dataset. Visible inputs: {glob.glob('/kaggle/input/*')}")
    return sorted(matches)[0]


def find_checkpoint_manifest():
    manifests = []
    for root, _, files in os.walk("/kaggle/input"):
        if "checkpoint_bundle_manifest.json" in files:
            manifests.append(os.path.join(root, "checkpoint_bundle_manifest.json"))
    if not manifests:
        raise FileNotFoundError(f"Could not find attached checkpoint manifest. Visible inputs: {glob.glob('/kaggle/input/*')}")
    manifest_path = sorted(manifests)[0]
    with open(manifest_path, "r", encoding="utf-8") as handle:
        manifest = json.load(handle)
    return manifest_path, manifest


CODE_DATASET_ROOT = find_code_dataset_root()
CHECKPOINT_MANIFEST_PATH, CHECKPOINT_MANIFEST = find_checkpoint_manifest()
CHECKPOINT_DATASET_ROOT = os.path.dirname(CHECKPOINT_MANIFEST_PATH)

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
shutil.copytree(CODE_DATASET_ROOT, WORKDIR)
for archive_name in ("staged_rl.tar", "tests.tar"):
    archive_path = os.path.join(WORKDIR, archive_name)
    if os.path.exists(archive_path):
        with tarfile.open(archive_path, "r") as tf:
            tf.extractall(WORKDIR)
        os.remove(archive_path)
        print("Extracted", archive_name)
for folder_name in ("staged_rl", "tests"):
    nested_path = os.path.join(WORKDIR, folder_name, folder_name)
    target_path = os.path.join(WORKDIR, folder_name)
    if os.path.isdir(nested_path):
        temp_path = f"{target_path}_flat"
        if os.path.exists(temp_path):
            shutil.rmtree(temp_path)
        shutil.move(nested_path, temp_path)
        shutil.rmtree(target_path)
        shutil.move(temp_path, target_path)
        print("Flattened", folder_name)

print("Using code dataset", CODE_DATASET_ROOT)
print("Using checkpoint dataset", CHECKPOINT_DATASET_ROOT)
print(json.dumps(CHECKPOINT_MANIFEST, indent=2))
print("Copied code to", WORKDIR)
print(sorted(os.listdir(WORKDIR)))


In [ ]:
subprocess.run(["python3", "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["python3", "-m", "uv", "venv", "--seed", "--clear", VENV_DIR], check=True)
install_commands = [
    [VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "--upgrade", "pip", "setuptools", "wheel"],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "numpy==1.26.4",
        "scipy==1.15.3",
        "scikit-learn==1.6.1",
    ],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "pillow",
        "packaging",
        "safetensors",
        "torchvision",
        "bitsandbytes",
        "xformers",
        "huggingface_hub>=0.34.0",
        "datasets==4.3.0",
        "transformers==4.56.2",
        "trl==0.22.2",
        "unsloth",
    ],
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "vllm==0.10.2",
        "--extra-index-url",
        "https://wheels.vllm.ai/0.10.2/",
    ],
]
for command in install_commands:
    print("Running:", " ".join(command))
    subprocess.run(command, check=True, cwd=WORKDIR)
print("Venv ready:", VENV_PYTHON)


In [ ]:
compat_script = r'''
import numpy
import scipy
import sklearn
import torch
import transformers
import trl
import unsloth
import vllm
print({
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "trl": trl.__version__,
    "vllm": vllm.__version__,
})
'''
subprocess.run([VENV_PYTHON, '-c', compat_script], check=True, cwd=WORKDIR)


In [ ]:
HARDWARE_PROFILE = "kaggle_t4"
TRAIN_SPLIT = "test"
EVAL_SPLIT = "testmini"
OUTPUT_ROOT = "outputs_staged_phase_d_from_large_phase_c"
MAX_EVAL_EXAMPLES_PER_SUBSET = 4
PHASE_RUNS = [
    {"phase": "phase_d", "resume": None, "warm_start": True},
]

selected_checkpoint = CHECKPOINT_MANIFEST["selected_checkpoint"]
WARM_START_PATH = os.path.join(CHECKPOINT_DATASET_ROOT, selected_checkpoint["checkpoint_relpath"])
if not os.path.exists(os.path.join(WARM_START_PATH, "adapter_model.safetensors")):
    raise FileNotFoundError(f"Warm-start adapter not found at {WARM_START_PATH}")

print("Warm-start checkpoint:", WARM_START_PATH)
print("Warm-start metrics:", json.dumps(selected_checkpoint.get("metrics", {}), indent=2))

env = dict(os.environ)
env["PYTHONUNBUFFERED"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

for run_spec in PHASE_RUNS:
    phase = run_spec["phase"]
    resume = run_spec["resume"]
    cmd = [
        VENV_PYTHON,
        "rl_gspo_qwen2_5vlm_test3.py",
        "--phase", phase,
        "--hardware-profile", HARDWARE_PROFILE,
        "--output-root", OUTPUT_ROOT,
        "--train-split", TRAIN_SPLIT,
        "--eval-split", EVAL_SPLIT,
        "--max-eval-examples-per-subset", str(MAX_EVAL_EXAMPLES_PER_SUBSET),
    ]
    if resume:
        cmd.extend(["--resume", resume])
    if run_spec["warm_start"]:
        cmd.extend(["--warm-start-checkpoint", WARM_START_PATH])

    phase_output_dir = os.path.join(WORKDIR, OUTPUT_ROOT, phase)
    os.makedirs(phase_output_dir, exist_ok=True)
    train_log_path = os.path.join(phase_output_dir, "train_subprocess.log")
    print("Running:", " ".join(cmd))
    print("Subprocess log:", train_log_path)
    with open(train_log_path, "w", encoding="utf-8") as log_file:
        completed = subprocess.run(
            cmd,
            check=False,
            cwd=WORKDIR,
            env=env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
        )
    if completed.returncode != 0:
        print(f"Phase {phase} failed with return code {completed.returncode}. Last 200 log lines:")
        with open(train_log_path, "r", encoding="utf-8") as log_file:
            tail_lines = log_file.readlines()[-200:]
        print("".join(tail_lines))
        raise subprocess.CalledProcessError(completed.returncode, cmd)
    print(f"Phase {phase} completed successfully.")


In [ ]:
outputs_root = os.path.join(WORKDIR, OUTPUT_ROOT)
collected = []
for root, _, files in os.walk(outputs_root):
    for file_name in files:
        collected.append(os.path.join(root, file_name))
for path in sorted(collected)[-80:]:
    print(path)
